# ASCII Maze RL — H100 GRPO Demo

GRPO training on 3×3 through 6×6 mazes using the SFT base model.
This notebook runs on an H100 GPU and is projected at the front of the room.

The SFT model solves 84% of 3×3, 30% of 4×4, and 4% of 5×5.
GRPO's job: push these higher through exploration and reward shaping.

In [ ]:
# Install deps
!pip install -q trl peft datasets accelerate bitsandbytes torchao 2>&1 | tail -1

import warnings, logging
warnings.filterwarnings('ignore')
logging.getLogger('torchao').setLevel(logging.ERROR)

import torch
if not torch.cuda.is_available():
    print("⚠️  CUDA not available! Restart runtime: Runtime → Restart session")
else:
    print(f"✓ CUDA OK: {torch.cuda.get_device_name(0)}")

In [ ]:
# Clone the maze infrastructure
!git clone -q -b claude-trl https://github.com/StephenJHardy/ascii-maze-rl.git
import sys
sys.path.insert(0, 'ascii-maze-rl')

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.mem_get_info(0)[1] / 1e9:.1f} GB")

## 1. Explore the Maze Format

Let's see what mazes look like and how solutions work.

In [ ]:
from src.maze_gen import generate
from src.maze_repr import to_str, to_prompt, SYSTEM_PROMPT, solution_to_str
from src.maze_verify import extract_moves, simulate, reached_exit, manhattan_progress
from src.reward import compute_reward

# Generate a sample maze
maze = generate(4, 4, seed=42)
print("Maze (4×4):")
print(to_str(maze))
print(f"\nSolution: {solution_to_str(maze.solution_moves)}")
print(f"Solution length: {len(maze.solution_moves)} moves")
print(f"\nEntry: {maze.entry}, Exit: {maze.exit}")

In [ ]:
# See how the prompt looks (this is what the model receives)
print("System prompt:")
print(SYSTEM_PROMPT)
print("\nUser message (just the maze grid):")
print(to_str(maze))

In [ ]:
# See how move parsing and simulation work
test_output = "r r d d d r"
moves = extract_moves(test_output)
print(f"Parsed moves: {moves}")

path = simulate(moves, maze)
print(f"Path through maze: {path}")
print(f"Reached exit: {reached_exit(path, maze)}")
print(f"Valid steps: {len(path) - 1} / {len(moves)}")
print(f"Progress toward exit: {manhattan_progress(path[-1], maze.exit, maze.entry):.2f}")

## 2. Generate Training & Evaluation Data

In [ ]:
from src.maze_dataset import MazeDataset, DatasetConfig, SizeConfig, MazeRecord

# GRPO training data: sizes where SFT has room to improve
train_config = DatasetConfig(
    name="grpo_train",
    algorithm="wilson",
    sizes=[
        SizeConfig(width=3, height=3, count=1000, start_seed=840_000),
        SizeConfig(width=4, height=4, count=2000, start_seed=850_000),
        SizeConfig(width=5, height=5, count=2000, start_seed=860_000),
        SizeConfig(width=6, height=6, count=2000, start_seed=870_000),
    ],
)
train_ds = MazeDataset.generate(train_config, progress=False)
print(f"Training: {train_ds.summary()}")

# Evaluation data (disjoint seeds)
eval_config = DatasetConfig(
    name="eval",
    algorithm="wilson",
    sizes=[
        SizeConfig(width=3, height=3, count=20, start_seed=900_000),
        SizeConfig(width=4, height=4, count=40, start_seed=910_000),
        SizeConfig(width=5, height=5, count=40, start_seed=920_000),
        SizeConfig(width=6, height=6, count=40, start_seed=930_000),
    ],
)
eval_ds = MazeDataset.generate(eval_config, progress=False)
print(f"Eval: {eval_ds.summary()}")

In [ ]:
from datasets import Dataset

# Rollout viewer config
ROLLOUT_SIZES = [
    SizeConfig(width=3, height=3, count=5, start_seed=970_000),
    SizeConfig(width=4, height=4, count=10, start_seed=980_000),
    SizeConfig(width=5, height=5, count=5, start_seed=990_000),
]
MAX_ROLLOUT_TOKENS = 60

def maze_records_to_hf_dataset(records):
    rows = []
    for r in records:
        prompt = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": r.maze_str},
        ]
        rows.append({
            "prompt": prompt,
            "maze_id": r.id,
            "solution_moves": r.solution_moves,
            "solution_length": r.solution_length,
            "walls": r.walls,
            "entry": r.entry,
            "exit": r.exit,
            "width": r.width,
            "height": r.height,
            "solution_path": r.solution_path,
        })
    return Dataset.from_list(rows)

train_hf = maze_records_to_hf_dataset(train_ds.records)
print(f"HF training dataset: {len(train_hf)} mazes")

## 3. Load the Pre-trained Model

We load `Qwen2.5-0.5B-Instruct` with a pre-trained SFT LoRA adapter
that already knows the maze format and can solve ~68% of 4×4 mazes.

Your GRPO reward function will push this higher.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "StephenJHardy/maze-cuda-sft-5000-qwen2.5-0.5b"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
)
print(f"Loaded SFT model from {model_id}")

## 4. Test the Model Before GRPO

Let's see how the model performs before RL training.

In [ ]:
def generate_solution(model, tokenizer, maze, max_new_tokens=60):
    """Generate a solution for a maze using the model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": to_str(maze)},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True,
    )
    return response


def evaluate_model(model, tokenizer, eval_dataset, max_new_tokens=60):
    """Quick evaluation — returns solve rate by size."""
    results = {}
    for record in eval_dataset:
        maze = record.to_maze()
        response = generate_solution(model, tokenizer, maze, max_new_tokens)
        moves = extract_moves(response)
        path = simulate(moves or [], maze)
        solved = reached_exit(path, maze)
        size = f"{record.width}x{record.height}"
        if size not in results:
            results[size] = {"total": 0, "solved": 0}
        results[size]["total"] += 1
        results[size]["solved"] += int(solved)
    return results


# Quick test on a few mazes
for seed in [42, 99, 7]:
    maze = generate(4, 4, seed=seed)
    response = generate_solution(model, tokenizer, maze)
    reward = compute_reward(response, maze)
    print(f"Maze 4x4_{seed}: output={response!r:40s} reward={reward:.3f}")

In [ ]:
# Full evaluation before GRPO
print("Evaluating model before GRPO...")
pre_results = evaluate_model(model, tokenizer, eval_ds.records)
print("\nPre-GRPO results:")
for size, stats in sorted(pre_results.items()):
    rate = stats['solved'] / stats['total'] * 100
    print(f"  {size}: {stats['solved']}/{stats['total']} ({rate:.0f}%)")

### Temperature & Rollout Exploration

Generate rollouts at different temperatures and store them.
Then re-score with different reward functions to see how
temperature and reward design interact.

In [ ]:
# Generate rollouts at multiple temperatures and store completions
# This only needs to run ONCE — then you can re-score with different rewards

import numpy as np
from src.maze_gen import generate
from src.maze_repr import to_str, SYSTEM_PROMPT
from src.maze_verify import extract_moves, simulate, reached_exit, manhattan_progress

# Sample 20 mazes across sizes
sample_mazes = (
    [generate(3, 3, seed=s) for s in range(900_000, 900_005)] +
    [generate(4, 4, seed=s) for s in range(910_000, 910_010)] +
    [generate(5, 5, seed=s) for s in range(920_000, 920_005)]
)

NUM_ROLLOUTS = 8
TEMPERATURES = [0.7, 1.0, 1.2, 1.5]

# Store all completions: {temp: {maze_idx: [completions]}}
stored_rollouts = {}

for temp in TEMPERATURES:
    stored_rollouts[temp] = {}
    for maze_idx, maze in enumerate(sample_mazes):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": to_str(maze)},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(model.device)

        completions = []
        for _ in range(NUM_ROLLOUTS):
            with torch.no_grad():
                out = model.generate(
                    **inputs, max_new_tokens=40, do_sample=True,
                    temperature=temp, pad_token_id=tokenizer.eos_token_id,
                )
            response = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
            completions.append(response)
        stored_rollouts[temp][maze_idx] = completions

    print(f"  temp={temp}: generated {len(sample_mazes)} x {NUM_ROLLOUTS} rollouts")

print(f"\nStored {len(TEMPERATURES)} x {len(sample_mazes)} x {NUM_ROLLOUTS} = "
      f"{len(TEMPERATURES)*len(sample_mazes)*NUM_ROLLOUTS} completions")
print("Now re-score with different reward functions below (instant, no GPU needed)")

In [ ]:
# Re-score stored rollouts with the CURRENT reward function
# Re-run this cell after changing your reward function — it's instant!

from src.maze_dataset import MazeRecord

print(f"{'Temp':>5s} | {'Mean Reward':>11s} | {'Std (variance)':>14s} | "
      f"{'Solved':>6s} | {'Parseable':>9s} | {'Zero-var mazes':>14s}")
print("-" * 80)

for temp in TEMPERATURES:
    all_rewards = []
    all_stds = []
    total_solved = 0
    total_parseable = 0
    total_rollouts = 0
    zero_var_count = 0

    for maze_idx, maze in enumerate(sample_mazes):
        completions = stored_rollouts[temp][maze_idx]
        record = MazeRecord.from_maze(maze)

        # Score with current maze_reward function
        kwargs = {
            "walls": [record.walls] * NUM_ROLLOUTS,
            "entry": [record.entry] * NUM_ROLLOUTS,
            "exit": [record.exit] * NUM_ROLLOUTS,
            "width": [record.width] * NUM_ROLLOUTS,
            "height": [record.height] * NUM_ROLLOUTS,
            "solution_path": [record.solution_path] * NUM_ROLLOUTS,
        }
        rewards = maze_reward(completions, **kwargs)

        mean_r = np.mean(rewards)
        std_r = np.std(rewards)
        all_rewards.append(mean_r)
        all_stds.append(std_r)
        if std_r < 0.005:
            zero_var_count += 1

        for comp, rew in zip(completions, rewards):
            total_rollouts += 1
            if extract_moves(comp) is not None:
                total_parseable += 1
            path = simulate(extract_moves(comp) or [], maze)
            if reached_exit(path, maze):
                total_solved += 1

    print(f"{temp:5.1f} | {np.mean(all_rewards):11.4f} | {np.mean(all_stds):14.4f} | "
          f"{total_solved:3d}/{total_rollouts:3d} | "
          f"{total_parseable:3d}/{total_rollouts:3d} | "
          f"{zero_var_count:3d}/{len(sample_mazes):3d}")

print("\n💡 Good reward functions have:")
print("   - High variance (std > 0.05) — gives GRPO signal")
print("   - Few zero-variance mazes — every maze teaches something")
print("   - Parseable > 90% — the model is outputting moves, not gibberish")

## 5. Reward Function

The reward function scores each rollout. It uses partial credit:
negative for gibberish, proportional to valid moves and progress
for partial solutions, and a bonus for reaching the exit.

In [ ]:
from src.maze_gen import Maze, solve as bfs_solve


def reconstruct_maze(walls, entry, exit_, width, height, solution_path):
    """Reconstruct a Maze object from dataset metadata."""
    frozen_walls = frozenset(
        frozenset(tuple(c) for c in w) for w in walls
    )
    return Maze(
        width=width,
        height=height,
        walls=frozen_walls,
        entry=tuple(entry),
        exit=tuple(exit_),
        solution=tuple(tuple(p) for p in solution_path),
        seed=0,
    )


def maze_reward(completions, **kwargs):
    """
    BFS-distance reward function for GRPO maze training.
    Uses actual maze distance (not manhattan) for partial credit.
    """
    rewards = []

    for i, completion in enumerate(completions):
        if isinstance(completion, list):
            completion = completion[-1]["content"] if completion else ""
        elif isinstance(completion, dict):
            completion = completion.get("content", "")

        maze = reconstruct_maze(
            walls=kwargs["walls"][i],
            entry=kwargs["entry"][i],
            exit_=kwargs["exit"][i],
            width=kwargs["width"][i],
            height=kwargs["height"][i],
            solution_path=kwargs["solution_path"][i],
        )

        moves = extract_moves(completion)

        if moves is None:
            rewards.append(-1.0)
            continue

        path = simulate(moves, maze)
        valid_steps = len(path) - 1
        optimal_len = len(maze.solution) - 1

        if reached_exit(path, maze):
            efficiency = optimal_len / max(len(moves), optimal_len)
            rewards.append(0.6 + 0.4 * efficiency)
            continue

        # BFS distance from final position to exit
        final_pos = path[-1]
        remaining = bfs_solve(maze.width, maze.height, maze.walls, final_pos, maze.exit)
        remaining_dist = len(remaining) - 1 if remaining else optimal_len
        bfs_progress = max(1.0 - remaining_dist / max(optimal_len, 1), 0.0)

        # Loop penalty
        unique_cells = len(set(path))
        loop_penalty = 1.0 - (len(path) - unique_cells) / max(len(path), 1)

        # Combine
        coverage = min(valid_steps / max(optimal_len, 1), 1.0)
        partial = 0.5 * (0.7 * bfs_progress + 0.2 * loop_penalty + 0.1 * coverage)
        rewards.append(partial)

    return rewards

In [ ]:
# Test your reward function on some examples
maze = generate(4, 4, seed=42)
correct = solution_to_str(maze.solution_moves)

test_completions = [
    correct,                    # perfect solution
    "r r d d",                  # wrong moves (may hit wall)
    correct + " u u u",         # correct + extra junk
    "I cannot solve this maze", # gibberish
    "",                         # empty
]

# Build kwargs as TRL would pass them
n = len(test_completions)
test_kwargs = {
    "walls": [MazeRecord.from_maze(maze).walls] * n,
    "entry": [list(maze.entry)] * n,
    "exit": [list(maze.exit)] * n,
    "width": [maze.width] * n,
    "height": [maze.height] * n,
    "solution_path": [[list(p) for p in maze.solution]] * n,
}

rewards = maze_reward(test_completions, **test_kwargs)
print("Reward function test:")
for comp, rew in zip(test_completions, rewards):
    print(f"  {rew:+.3f}  {comp!r:.50s}")

print("\n💡 If all non-solved outputs score the same, GRPO won't learn much.")
print("   Try adding partial credit so 'almost right' scores higher than 'gibberish'.")

## 6. Train with GRPO

Now we run GRPO using your reward function. The trainer will:
1. Sample prompts from the training set
2. Generate multiple completions per prompt
3. Score them with your reward function
4. Update the model toward higher-reward completions

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from peft import LoraConfig
from transformers import TrainerCallback

grpo_config = GRPOConfig(
    loss_type="grpo",              # explicit — TRL defaults to "dapo" now
    output_dir="./maze-grpo-output",
    num_train_epochs=1,
    max_steps=10000,
    per_device_train_batch_size=16,
    num_generations=16,
    max_completion_length=100,
    learning_rate=1e-5,
    beta=0.04,
    logging_steps=50,
    save_steps=1000,
    temperature=1.0,
    log_completions=False,
    report_to="none",
    bf16=False,
    fp16=True,
    gradient_accumulation_steps=1,
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

class RewardLogger(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            keys = [k for k in logs if 'reward' in k]
            if keys:
                parts = [f"{k}={logs[k]:.4f}" for k in keys]
                print(f"  Step {state.global_step}: {' | '.join(parts)}")

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[maze_reward],
    args=grpo_config,
    train_dataset=train_hf,
    peft_config=peft_config,
    callbacks=[RewardLogger()],
)

print("Starting GRPO training...")
print(f"  Steps: {grpo_config.max_steps}")
print(f"  Generations per prompt: {grpo_config.num_generations}")
print(f"  Max completion tokens: {grpo_config.max_completion_length}")
print(f"  LR: {grpo_config.learning_rate}, beta: {grpo_config.beta}")

In [ ]:
trainer.train()

## Explore Rollouts

Capture GRPO-style rollouts and visualize them. This shows what
GRPO sees at each step: multiple attempts at the same maze,
their rewards, and which ones get reinforced.

In [ ]:
from src.rollout_capture import capture_rollouts_pytorch, save_rollouts
from src.maze_dataset import MazeDataset, DatasetConfig, SizeConfig

# Generate a small eval set for rollout visualization
rollout_config = DatasetConfig(
    name="rollout_viz",
    algorithm="wilson",
    sizes=ROLLOUT_SIZES,
)
rollout_ds = MazeDataset.generate(rollout_config, progress=False)

# Capture rollouts from the trained model
print("Capturing rollouts from trained model...")
trained_model = trainer.model
trained_model.eval()
rollouts = capture_rollouts_pytorch(
    trained_model, tokenizer, rollout_ds.records,
    num_generations=8, temperature=1.0, max_new_tokens=MAX_ROLLOUT_TOKENS,
)
save_rollouts(rollouts, 'rollouts.json')
print(f"Captured {sum(len(r.rollouts) for r in rollouts)} rollouts across {len(rollouts)} mazes")

In [ ]:
# Build and display the rollout viewer
from src.build_rollout_viewer import HTML_TEMPLATE
import json as _json
from dataclasses import asdict
from IPython.display import HTML, display

viewer_data = [asdict(r) for r in rollouts]
serialized = _json.dumps(viewer_data).replace('</','<\\/')
viewer_html = HTML_TEMPLATE.replace('__DATA_PLACEHOLDER__', serialized)

# Save to file (downloadable) and display inline
with open('rollout_viewer.html', 'w') as f:
    f.write(viewer_html)
print('Viewer saved to rollout_viewer.html')

# Display inline (limited height)
display(HTML(f'<iframe srcdoc="{viewer_html.replace(chr(34), "&quot;")}" width="100%" height="800px" style="border:none;"></iframe>'))

## 7. Evaluate After GRPO

In [ ]:
# Evaluate the trained model
trained_model = trainer.model

print("Evaluating model after GRPO...")
post_results = evaluate_model(trained_model, tokenizer, eval_ds.records)

print("\nResults comparison:")
print(f"{'Size':>5s}  {'Before':>8s}  {'After':>8s}  {'Change':>8s}")
print("-" * 35)
for size in sorted(set(list(pre_results.keys()) + list(post_results.keys()))):
    pre = pre_results.get(size, {"solved": 0, "total": 0})
    post = post_results.get(size, {"solved": 0, "total": 0})
    pre_rate = pre["solved"] / max(pre["total"], 1) * 100
    post_rate = post["solved"] / max(post["total"], 1) * 100
    diff = post_rate - pre_rate
    print(f"{size:>5s}  {pre_rate:7.1f}%  {post_rate:7.1f}%  {diff:+7.1f}pp")

In [ ]:
# What is the model actually outputting now?
print("Sample outputs after GRPO:\n")
for record in eval_ds.records[:10]:
    maze = record.to_maze()
    response = generate_solution(trained_model, tokenizer, maze)
    print(f"  {record.id}: {response!r:.60s}")

print("\n" + "="*60)
print("If results got WORSE, your reward function likely has a problem.")
print("Common issues with the naive binary reward (1.0/0.0):")
print("  - All 8 generations score 0.0 → advantages are all zero")
print("  - GRPO gets no gradient signal → model drifts randomly")
print("  - Over many steps, drift destroys the SFT baseline")
print("")
print("Fix: Go back to Section 5 and add PARTIAL CREDIT:")
print("  - Negative reward for gibberish (not zero!)")
print("  - Partial credit proportional to valid moves")
print("  - Progress toward exit as a smooth signal")
print("="*60)

In [ ]:
# Look at some specific examples
print("\nSample outputs after GRPO:\n")
for seed in [42, 99, 7, 123, 456]:
    for size in [4, 5]:
        maze = generate(size, size, seed=seed)
        response = generate_solution(trained_model, tokenizer, maze)
        moves = extract_moves(response)
        path = simulate(moves or [], maze)
        solved = reached_exit(path, maze)
        status = "✓" if solved else "✗"
        correct = solution_to_str(maze.solution_moves)
        print(f"  {status} {size}x{size}_{seed}: model={response!r:30s} correct={correct}")